# 환경 설정

In [ ]:
!pip install -U langchain langchain-core langchain-google-genai cohere

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
# 코랩 Secrets에서 GOOGLE_API_KEY를 가져오기

google_api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = google_api_key


In [ ]:
import cohere
api_key = userdata.get('COHERE_API_KEY')
co = cohere.Client(api_key)

# 데이터

## 소설 데이터 불러오기

In [ ]:
import huggingface_hub
from datasets import load_dataset
import pandas as pd

#허깅페이스에서 소설 데이터 불러오기
huggingface_hub.login(token="YOUR_TOKEN")
df = load_dataset('werty1248/Korean-1930-Novel-Scene-Summarize')
df = pd.DataFrame(df['train'])

README.md:   0%|          | 0.00/739 [00:00<?, ?B/s]

ko_text.json: reconstructing file:   0%|          |  0.00B / 25.1MB            

ko_text.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12108 [00:00<?, ? examples/s]

In [ ]:
#리스트 형태로 저장
ids = df['id'][:600]
titles = []
contents = []
i_list = []
index = []
content = ''
novel_count = 0 #각 소설 회차수
total_count = 0 #전체 소설 개수
for i in range(len(ids)):
  title = ids[i][:-5].split('-')
  content += df['text'][i]
  index.append(i)
  if total_count < 3:
    title = title[-1]
  else:
    title = title[-2]
  if i % 6 == 5:
    if len(index) == 1:
      continue
    contents.append(content)
    i_list.append(index)
    content = ''
    index = []
    novel_count += 1
    titles.append(f'{title.replace('_', ' ').strip()} {novel_count}화')
  if i == len(ids) -1 or ids[i+1].find(title) == -1:
    if i % 6 != 5:
      contents[-1] += content
      i_list[-1].extend(index)
    content = ''
    index = []
    novel_count = 0
    total_count += 1

In [ ]:
titles[:10]

['백치(白痴) 아다다 1화',
 '백치(白痴) 아다다 2화',
 '백치(白痴) 아다다 3화',
 '백치(白痴) 아다다 4화',
 '며느리 1화',
 '며느리 2화',
 '며느리 3화',
 '며느리 4화',
 '죄와벌 1화',
 '죄와벌 2화']

## 소설 질문 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
with open("/content/drive/MyDrive/[파일 경로]/novel_question.txt", "r", encoding="utf-8") as file:
  q_list = file.readlines()
q_list[:5]

['아다다가 넘어져 있던 곳은 어디이며, 그때 함께 깨져 있던 것은 무엇인가?\n',
 '어머니가 처음 딸의 사고를 발견했을 때 더 안타까워한 것은 딸의 부상인가, 깨진 그릇인가?\n',
 "아다다의 본명은 무엇이며, 그녀가 '아다다'라는 이름으로 불리게 된 이유는 무엇인가?\n",
 '아다다가 벙어리가 된 것과 별개로, 그녀의 성격적 특징(둔한 지혜, 힘에 부치는 일을 굳이 하려는 습성)은 어떻게 묘사되는가?\n',
 '아다다는 몇 살에 시집을 갔으며, 시집갈 때 친정에서 무엇을 함께 보냈는가?\n']

## 소설 요약본 생성하기

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sqlite3
from tqdm import tqdm
import pandas as pd

model = ChatGoogleGenerativeAI(model="gemma-4-26b-a4b-it")
print("\n--- 대화형 챗봇 시작 ---")
novel_data = [titles[i]+'\n'+contents[i] for i in range(len(contents))]
print(novel_data[0][:40])
summary = []
for c in tqdm(novel_data):
      answer = ''
      #소설 요약
      prompt = ChatPromptTemplate.from_template(
        """

        당신은 친절한 한국어 챗봇입니다.다음 소설 내용을 읽고 요약해주세요.

        #소설 내용: {novel}
        """
        )
      chain = prompt | model | StrOutputParser()
      answer = chain.invoke({'novel': c})

      summary.append(answer)

#csv 파일로 저장
data = {"title":titles, "contents": contents, "summary":summary}
data = pd.DataFrame(data)
data.to_csv("/content/drive/MyDrive/[파일 경로]/novel_summary.csv", index=False)

# 하이브리드 검색

## 의미 검색

In [ ]:
from sentence_transformers import SentenceTransformer
#sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
#sentence_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
sentence_model = SentenceTransformer('intfloat/multilingual-e5-base')
embeddings_s = sentence_model.encode(["passage: " + c for c in summary]) #요약문 임베딩
embeddings_s.shape

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

(97, 768)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def dense_vector_search(query, contents, embeddings, k):
    emd = sentence_model.encode("query: " + query) #임베딩
    sim_scores = [cosine_similarity([embeddings[i]], [emd]) for i in range(len(contents))]
    index = range(0, len(contents))
    pairs = zip(sim_scores, index)
    result = sorted(pairs, reverse=True)[:k]
    return zip(*result)


## 키워드 검색

In [ ]:
import math
import numpy as np
from typing import List
from transformers import PreTrainedTokenizer, AutoTokenizer
from collections import defaultdict

class BM25:
  def __init__(self, corpus:List[List[str]], tokenizer:PreTrainedTokenizer):
    self.tokenizer = tokenizer
    self.corpus = corpus
    self.tokenized_corpus = self.tokenizer(corpus, add_special_tokens=False)['input_ids']
    self.n_docs = len(self.tokenized_corpus)
    self.avg_doc_lens = sum(len(lst) for lst in self.tokenized_corpus) / len(self.tokenized_corpus)
    self.idf = self._calculate_idf()
    self.term_freqs = self._calculate_term_freqs()

  def _calculate_idf(self):
    idf = defaultdict(float)
    for doc in self.tokenized_corpus:
      for token_id in set(doc):
        idf[token_id] += 1
    for token_id, doc_frequency in idf.items():
      idf[token_id] = math.log(((self.n_docs - doc_frequency + 0.5) / (doc_frequency + 0.5)) + 1)
    return idf

  def _calculate_term_freqs(self):
    term_freqs = [defaultdict(int) for _ in range(self.n_docs)]
    for i, doc in enumerate(self.tokenized_corpus):
      for token_id in doc:
        term_freqs[i][token_id] += 1
    return term_freqs

  def get_scores(self, query:str, k1:float = 1.2, b:float=0.75):
    query = self.tokenizer([query], add_special_tokens=False)['input_ids'][0]
    scores = np.zeros(self.n_docs)
    for q in query:
      idf = self.idf[q]
      for i, term_freq in enumerate(self.term_freqs):
        q_frequency = term_freq[q]
        doc_len = len(self.tokenized_corpus[i])
        score_q = idf * (q_frequency * (k1 + 1)) / ((q_frequency) + k1 * (1 - b + b * (doc_len / self.avg_doc_lens)))
        scores[i] += score_q
    return scores

  def get_top_k(self, query:str, k:int):
    scores = self.get_scores(query)
    top_k_indices = np.argsort(scores)[-k:][::-1]
    top_k_scores = scores[top_k_indices]
    return top_k_scores, top_k_indices

b_tokenizer = AutoTokenizer.from_pretrained('klue/roberta-base')
bm25 = BM25(contents, b_tokenizer)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1567 > 512). Running this sequence through the model will result in indexing errors


## 하이브리드 검색

In [ ]:
##단순 합
def new_rank(scores:List[List[int]], rankings):
    rrf = defaultdict(float)
    for i in range(len(scores[0])):
            rrf[rankings[0][i]] +=  scores[0][i]
            rrf[rankings[1][i]] +=  scores[1][i]
    return sorted(rrf, key=rrf.get, reverse=True)

In [ ]:
##비율 합산
def new_rank_b(scores:List[List[int]], rankings):
    rrf = defaultdict(float)
    for i in range(len(scores[0])):
            rrf[rankings[0][i]] +=  scores[0][i] * 0.1
            rrf[rankings[1][i]] +=  scores[1][i] * 0.9
            #print(i, s)
    return sorted(rrf, key=rrf.get, reverse=True)

In [ ]:
##동적 합산
def new_rank_d(r, scores:List[List[int]], rankings):
    rrf = defaultdict(float)
    for i in range(len(scores[0])):
            rrf[rankings[0][i]] +=  scores[0][i] * r
            rrf[rankings[1][i]] +=  scores[1][i] * (1-r)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)

In [ ]:
## 하이브리드 검색
def hybrid_search(query, contents, embeddings, bm25, x, k, type):
  d_scores, dense_search_ranking = dense_vector_search(query, contents, embeddings, k)#의미 검색
  b_scores, bm25_search_ranking = bm25.get_top_k(query, k) # bm25(키워드) 검색
  results = []

  if type == 0: #type이 0일 때 단순 합
    for i in range(len(b_scores)):
      b_scores[i] /=  x #단순 합은 bm25를 k로 나누고 계산
    results = new_rank(scores = [d_scores, b_scores], rankings=[dense_search_ranking, bm25_search_ranking])
    return results

  #비율 합산과 동적 비율 합산은  bm25 정규화 후 계산
  for i in range(len(b_scores)):
    b_scores[i] /=  (x + b_scores[i])
  if type == 1: #type이 1일 때 비율 합산
    results = new_rank_b(scores = [d_scores, b_scores], rankings=[dense_search_ranking, bm25_search_ranking])
  else: #type이 2일 때 동적 비율 합산
    results = new_rank_b(scores = [d_scores, b_scores], rankings=[dense_search_ranking, bm25_search_ranking])
  return results

In [ ]:
## 하이브리드 검색 테스트(파라미터 설정)_단순 합 기준
from tqdm import tqdm
x_list = [2, 4, 6] # 단순 합일 땐 bm25/x, 나머지는 bm25 / (x + bm25)
k_list = [20, 25, 30, 35] #각 검색 결과 반환 개수
print("단순 합")
for x in x_list:
  for k in k_list:
    hit_at_3 = 0
    hit_at_10 = 0
    hit_at_15 = 0
    hit_at_20 = 0
    print(f"x={x}, 반환개수={n}")
    for i in tqdm(range(len(q_list))):
      results = hybrid_search(q_list[i], contents, embeddings_s, bm25, x, k, 0)
      if (i//10) in results[:20]:
        hit_at_20 += 1
        if (i//10) in results[:15]:
          hit_at_15 += 1
          if (i//10) in results[:10]:
            hit_at_10 += 1
            if (i//10) in results[:3]:
              hit_at_3 += 1
    print("hit@3: ", hit_at_3)
    print("hit@10: ", hit_at_10)
    print("hit@15: ", hit_at_15)
    print("hit@20: ", hit_at_20)

## 재순위화

In [ ]:
def rerank(query: str, ids: list, top_k: int = 3) -> list:
    docs = [contents[i] for i in ids]

    response = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=docs,
    top_n=top_k  # 상위 3개 반환
    )
    results = response.results
    return [ids[x.index] for x in results]

In [ ]:
## 하이브리드 검색으로 맞힌 것들(15개 이내에 정답 있는 것들)
ids = []
for i in tqdm(range(len(q_list))):
  results = hybrid_search(q_list[i], contents, embeddings_s, bm25, 10, 30, 1) #비율 합산, 최종 선택된 파라미터들
  if (i//10) in results[:15]:
      ids.append(i)

100%|██████████| 970/970 [08:59<00:00,  1.80it/s]


In [ ]:
import random
## 샘플링
random.seed(42)
ids = random.sample(ids, 200)
ids[:5]

[662, 114, 25, 771, 285]

In [ ]:
import time
##하이브리드+재순위화 테스트 (200개)
score = 0
for i in ids:
  candidates = hybrid_search(q_list[i], contents, embeddings_s, bm25, 10, 30)[:15]
  results = rerank(q_list[i], candidates, 3)
  answer = i//10
  time.sleep(4.9)
  if (i // 10) in results:
    score += 1
print(len(score))

193
